In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/city_day.csv')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (29531, 16)
Columns: ['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [2]:
# 1. See which cities we have and how many rows for each
print("Total rows:", len(df))
print(df['City'].value_counts())

Total rows: 29531
City
Ahmedabad             2009
Bengaluru             2009
Chennai               2009
Delhi                 2009
Lucknow               2009
Mumbai                2009
Hyderabad             2006
Patna                 1858
Gurugram              1679
Visakhapatnam         1462
Amritsar              1221
Jorapokhar            1169
Jaipur                1114
Thiruvananthapuram    1112
Amaravati              951
Brajrajnagar           938
Talcher                925
Kolkata                814
Guwahati               502
Coimbatore             386
Shillong               310
Chandigarh             304
Bhopal                 289
Ernakulam              162
Kochi                  162
Aizawl                 113
Name: count, dtype: int64


In [3]:
# 2. Filter to our 3 focus cities
cities = ['Delhi', 'Mumbai', 'Bengaluru']
df_filtered = df[df['City'].isin(cities)].copy()

# 3. How many rows do we have now?
print("Rows for our 3 cities:", len(df_filtered))

# 4. Look at missing values in PM2.5 for these cities
print("\nMissing PM2.5 values before cleaning:")
print(df_filtered['PM2.5'].isna().sum())

Rows for our 3 cities: 6027

Missing PM2.5 values before cleaning:
1373


In [4]:
# 5. Drop rows where PM2.5 is missing (we can't predict what we don't know)
df_clean = df_filtered.dropna(subset=['PM2.5']).copy()

print("Rows left after dropping missing PM2.5:", len(df_clean))

# 6. Convert the 'Date' column to actual datetime objects so we can use time-series later
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# 7. Let's look at Delhi's first 5 clean rows
df_clean[df_clean['City'] == 'Delhi'].head()

Rows left after dropping missing PM2.5: 4654


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
10229,Delhi,2015-01-01,313.22,607.98,69.16,36.39,110.59,33.85,15.20,9.25,41.68,14.36,24.86,9.84,472.0,Severe
10230,Delhi,2015-01-02,186.18,269.55,62.09,32.87,88.14,31.83,9.54,6.65,29.97,10.55,20.09,4.29,454.0,Severe
10231,Delhi,2015-01-03,87.18,131.90,25.73,30.31,47.95,69.55,10.61,2.65,19.71,3.91,10.23,1.99,143.0,Moderate
10232,Delhi,2015-01-04,151.84,241.84,25.01,36.91,48.62,130.36,11.54,4.63,25.36,4.26,9.71,3.34,319.0,Very Poor
10233,Delhi,2015-01-05,146.60,219.13,14.01,34.92,38.25,122.88,9.20,3.33,23.20,2.80,6.21,2.96,325.0,Very Poor


In [5]:
# 1. Save the clean data to a new CSV file
df_clean.to_csv('../data/clean/city_day_clean.csv', index=False)
print("Saved clean data to data/clean/city_day_clean.csv")

# 2. Set up SQL inside our notebook using SQLite
import sqlite3
conn = sqlite3.connect(':memory:') # Creates a temporary database in RAM
df_clean.to_sql('air_quality', conn, index=False, if_exists='replace')
print("Data loaded into SQL table 'air_quality'")

# 3. Helper function to run SQL and show results nicely
def run_sql(query):
    return pd.read_sql_query(query, conn)

Saved clean data to data/clean/city_day_clean.csv
Data loaded into SQL table 'air_quality'


In [6]:
query1 = """
SELECT City, 
       ROUND(AVG("PM2.5"), 2) as avg_pm25,
       MAX("PM2.5") as max_pm25
FROM air_quality
GROUP BY City
ORDER BY avg_pm25 DESC;
"""
run_sql(query1)

,City,avg_pm25,max_pm25
0,Delhi,117.20,685.36
1,Bengaluru,35.82,313.59
2,Mumbai,35.20,134.32


In [7]:
query2 = """
SELECT Date, "PM2.5", AQI_Bucket
FROM air_quality
WHERE City = 'Delhi'
ORDER BY "PM2.5" DESC
LIMIT 5;
"""
run_sql(query2)

,Date,PM2.5,AQI_Bucket
0,2016-11-06 00:00:00,685.36,Severe
1,2017-11-08 00:00:00,639.19,Severe
2,2016-11-05 00:00:00,588.39,Severe
3,2019-11-03 00:00:00,582.28,Severe
4,2016-11-02 00:00:00,537.96,Severe
